# Suite CON — Predicates

A predicate is a named Basic rule over symbols, each bound to an object of a schema; it applies to a match, which binds
every symbol. Predicates and their sets are built fluently, as schemas are, and are data: written by their schemas'
names, and read back through a store that resolves them.

In [ ]:
from mbse.Expressions import Expressions as E
from mbse.Expressions.Dialects.Python import Text
from mbse.Expressions.Framework import Terms
from mbse.Patterns import Constraints as C, register
from mbse.Schemas.Framework import JSON, Proxies, Schemas as S, Validators as SV
from support import Contact, Phone, book, raises

the = E.variable("the")
IsAnAdult = (
    C.OfPredicate.Builder()
    .name("IsAnAdult")
    .description("18 or older")
    .symbols({"the": Contact})
    .rule(Text.FromFunction(lambda the: the.age >= 18))
    .create()
)
Numbered = C.OfPredicate.Builder().name("Numbered").symbols({"the": Phone}).rule(the.has("number")).create()

## CON-01 · A predicate is built fluently: a name, a description, symbols and a rule

In [ ]:
assert (IsAnAdult.name, IsAnAdult.description, list(IsAnAdult.symbols)) == ("IsAnAdult", "18 or older", ["the"])
assert IsAnAdult.symbols["the"] is Contact and isinstance(IsAnAdult.rule, E.OfOperation.Data)  # read from the lambda
assert Terms.same(IsAnAdult.rule, the.age.ge(18).data)  # the same rule as the writer builds
assert Numbered.description is None and IsAnAdult.validate() == []
two = C.OfPredicate.Builder().symbols({"a": Contact}).symbols({"b": Phone}).create()
assert list(two.symbols) == ["a", "b"] and two.rule is None  # symbols add up, in order; nothing validates yet
renamed = C.OfPredicate.Builder(IsAnAdult).name("IsGrown").clone()
assert renamed.name == "IsGrown" and IsAnAdult.name == "IsAnAdult" and renamed.rule is IsAnAdult.rule
assert C.OfPredicate.Builder(renamed).description("grown").update() is renamed and renamed.description == "grown"
assert C.OfPredicate.resolve(IsAnAdult) is IsAnAdult
assert C.OfPredicate.resolve(lambda b: b.name("X").symbols({"x": Contact}).rule(True)).name == "X"
with raises(TypeError, match="expected a predicate or a callable taking its builder, got 'X'"):
    C.OfPredicate.resolve("X")
with raises(ValueError, match="create() is only valid without a source instance; use clone() or update()"):
    C.OfPredicate.Builder(IsAnAdult).create()
with raises(ValueError, match="clone() is only valid with a source instance"):
    C.OfPredicate.Builder().clone()
with raises(ValueError, match="update() is only valid with a source instance"):
    C.OfPredicate.Builder().update()
assert (IsAnAdult.schema_name(), IsAnAdult.owner(), IsAnAdult.identity() == IsAnAdult.identity()) == (
    "Patterns.Predicate", None, True)

## CON-02 · Predicates are checked when asked

In [ ]:
Value = S.OfObject.Builder().name("Value").create()
bad = C.OfPredicate.Builder().name("Bad").symbols({"a": Value, "b": S.OfObject.Builder().ref().create()}).rule(
    E.operation("pow", E.variable("a"), E.variable("n"))).create()
assert bad.validate() == ["predicate 'Bad': symbol 'a' needs a named reference object schema",
                          "predicate 'Bad': symbol 'b' needs a named reference object schema",
                          "predicate 'Bad': 'pow' is not a core operation",
                          "predicate 'Bad': argument 1: variable 'n' is not bound"]
assert C.OfPredicate.Builder().create().validate() == ["predicate None: a predicate needs a name",
                                                       "predicate None: a predicate needs a rule"]  # symbols are optional
twice = C.OfSet.Builder().predicates(IsAnAdult, Numbered, C.OfPredicate.Builder(IsAnAdult).clone()).create()
assert twice.validate() == ["predicate 'IsAnAdult': defined twice"]
with raises(ValueError, match="predicate 'Bad': symbol 'a' needs a named reference object schema; predicate 'Bad'"):
    C.check([bad])
predicates = C.check([IsAnAdult, lambda b: b.name("Named").symbols({"the": Contact}).rule(the.has("name"))])
assert isinstance(predicates, C.OfSet.Data) and C.check(predicates) is predicates

## CON-03 · A set orders predicates

In [ ]:
predicates = C.OfSet.Builder().predicates(IsAnAdult).predicates(Numbered).create()
assert predicates.predicates == (IsAnAdult, Numbered)  # a tuple, in order
assert C.OfSet.Builder().create().predicates == () and predicates.schema_name() == "Patterns.Set"
assert predicates.owner() is None and predicates.identity() == predicates.identity() != C.OfSet.Builder().create().identity()
grown = C.OfSet.Builder(predicates).predicates(lambda b: b.name("Grown").symbols({"the": Contact}).rule(True)).clone()
assert [p.name for p in grown.predicates] == ["IsAnAdult", "Numbered", "Grown"] and len(predicates.predicates) == 2

## CON-04 · Predicates are data: written by their symbols' schema names, read back through a store

In [ ]:
shared = the.age.ge(18).data
predicates = C.OfSet.Builder().predicates(
    C.OfPredicate.Builder(IsAnAdult).rule(shared).clone(),
    lambda b: b.name("Owns").symbols({"c": Contact, "p": Phone}).rule(E.operation("implies", shared, E.variable("p").has("number"))),
).create()
store = C.OfStore(book())  # resolves the symbols' schemas in the address book
text = JSON.ToJSON(store).Reachable(C.OfSet.Schema, predicates)
assert '"symbols": [{"name": "c", "type": {"named": {"name": "Contact"}}}' in text  # by name: no store needed to write
assert text.count('"kind": "operation"') == 4  # get, ge, implies and has: the shared rule's two are written once
assert JSON.ToJSON(C.Builders).Reachable(C.OfSet.Schema, predicates) == text  # any store writes them alike
copy = JSON.FromJSON(store).Reachable(C.OfSet.Schema, text)
assert [(p.name, p.description) for p in copy.predicates] == [("IsAnAdult", "18 or older"), ("Owns", None)]
assert copy.predicates[1].symbols["c"] is Contact and copy.predicates[1].symbols["p"] is Phone  # the store's own
assert copy.predicates[1].rule.arguments[0] is copy.predicates[0].rule  # still shared
assert Terms.same(copy.predicates[1].rule, predicates.predicates[1].rule)
assert SV.Validate(store).Reachable(C.OfSet.Schema, predicates) == []  # valid data, by mbse-schemas' rules
with raises(LookupError, match="no schema named 'Contact' in the module or the store"):
    JSON.FromJSON(C.Builders).Reachable(C.OfSet.Schema, text)  # reading resolves names, so it needs the store
Inline = S.OfObject.Builder().ref().properties(lambda p: p.name("n").of(lambda t: t.as_native(int))).create()
loose = C.OfPredicate.Builder().name("Loose").symbols({"x": Inline}).rule(True).create()
back = JSON.FromJSON(C.Builders).Reachable(C.OfPredicate.Schema, JSON.ToJSON(C.Builders).Reachable(C.OfPredicate.Schema, loose))
assert back.symbols["x"] is not Inline and list(back.symbols["x"].properties) == ["n"]  # an unnamed schema, inline
built = store.builder("Patterns.Predicate", IsAnAdult)
built.property("name", lambda p: p.value(lambda a: a.as_native(lambda n: n.set("Grown"))))
assert built.update() is IsAnAdult and IsAnAdult.name == "Grown" and IsAnAdult.symbols["the"] is Contact
IsAnAdult.name = "IsAnAdult"

## CON-05 · Decoding refuses what a predicate or a set cannot hold

In [ ]:
store = C.OfStore(book())


def index(e, i):
    e.property("index", lambda p: p.value(lambda v: v.as_native(lambda n: n.set(i))))


def rule_entry(target, i=0):
    def fill(e):
        if target is not None:
            e.link("argument", lambda l: l.set(target))
        index(e, i)
    return lambda a: a.add(fill)


def member_entry(target, i=None):
    def fill(e):
        if target is not None:
            e.link("predicate", lambda l: l.set(target))
        if i is not None:
            index(e, i)
    return lambda a: a.add(fill)


with raises(ValueError, match="a predicate has one rule, got 2"):
    store.builder("Patterns.Predicate").adjacency("rule", rule_entry(the.data)).adjacency("rule", rule_entry(the.data, 1)).create()
with raises(TypeError, match="a rule must be an expression"):
    store.builder("Patterns.Predicate").adjacency("rule", rule_entry(Numbered)).create()
with raises(ValueError, match="link 'argument' is not set"):
    store.builder("Patterns.Predicate").adjacency("rule", rule_entry(None)).create()
with raises(TypeError, match="a member must be a predicate"):
    store.builder("Patterns.Set").adjacency("predicates", member_entry(the.data, 0)).create()
with raises(ValueError, match="link 'predicate' is not set"):
    store.builder("Patterns.Set").adjacency("predicates", member_entry(None, 0)).create()
unordered = store.builder("Patterns.Set").adjacency("predicates", member_entry(Numbered)).adjacency(
    "predicates", member_entry(IsAnAdult, 0)).adjacency("predicates", member_entry(IsAnAdult)).create()
assert unordered.predicates == (IsAnAdult, Numbered, IsAnAdult)  # entries without an index come last, in their order
blank = store.builder("Patterns.Predicate").create()  # what a snapshot leaves out is absent, and validate() says so
assert (blank.name, blank.description, blank.symbols, blank.rule) == (None, None, {}, None)
assert JSON.ToJSON(C.Builders)(C.OfPredicate.Schema, blank) == '{"root": "s0", "objects": {"s0": {}}}'  # nothing to write

## CON-06 · Another store holds predicates as proxies

In [ ]:
store = register(Proxies.OfStore())
assert {"Patterns.Predicate", "Patterns.Set", "Patterns.Members", "Expressions.OfOperation"} <= set(store.names())
assert register(store) is store  # what it holds already is skipped
loose = C.OfPredicate.Builder().name("Any").symbols({"x": S.OfObject.Builder().ref().create()}).rule(True).create()
proxy = JSON.FromJSON(store).Reachable(C.OfPredicate.Schema, JSON.ToJSON(C.Builders).Reachable(C.OfPredicate.Schema, loose))
assert proxy.schema_name() == "Patterns.Predicate" and Proxies.store_of(proxy) is store and proxy.name == "Any"